# EXACT 2026 — serve SFT'd Qwen2.5-7B+LoRA as an OpenAI endpoint (free Colab T4)

Loads base **Qwen2.5-7B-Instruct (4-bit)** + your trained LoRA adapter,
exposes a minimal OpenAI-compatible `/v1/chat/completions` over a
**cloudflared** public URL (no signup). Point the local repo's
`configs/model.yaml::llm.vllm_base_url` at it and run
`scripts/run_eval.py --with-llm`.

**Use a CLEAN runtime** (Runtime > Disconnect and delete runtime if you
reused this session). GPU = T4. Keep the LAST cell running — the tunnel
lives only while it runs.

## 1. Install (unsloth self-pins ML stack; +tiny web shim)

In [ ]:
%%capture
# Minimal. unsloth pins the consistent ML stack; fastapi/uvicorn are
# pure-web and don't touch it. cloudflared = static binary, no pip.
!pip install -q unsloth
!pip install -q fastapi uvicorn nest-asyncio
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
print(">>> If you reused a runtime and hit errors: Runtime > Disconnect and delete runtime, then Run all. <<<")

## 2. Upload the adapter zip

Run this, pick `exact_qwen25_7b_lora.zip` (the file you downloaded from
the training notebook, on your machine at `D:\Exact2026\`).

In [ ]:
import os, zipfile
ADAPTER_DIR = "exact_qwen25_7b_lora"
if not os.path.isdir(ADAPTER_DIR):
    from google.colab import files
    up = files.upload()  # choose exact_qwen25_7b_lora.zip
    zname = next(k for k in up if k.endswith('.zip'))
    with zipfile.ZipFile(zname) as z:
        z.extractall('.')
assert os.path.exists(f"{ADAPTER_DIR}/adapter_config.json"), "adapter not found"
print("adapter ok:", os.listdir(ADAPTER_DIR))

## 3. Load base + adapter (Unsloth reads base from adapter_config)

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048
# adapter_config.json points at the base; Unsloth loads base 4-bit + LoRA.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="exact_qwen25_7b_lora",
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    dtype=None,
)
FastLanguageModel.for_inference(model)  # 2x faster decode
print("loaded; device:", next(model.parameters()).device)

## 4. Minimal OpenAI-compatible shim

Implements exactly what the repo's `VLLMClient` (chat mode) calls:
`POST /v1/chat/completions` → `{choices:[{message:{content}}]}`,
plus `GET /healthz`. Runs uvicorn in a daemon thread.

In [ ]:
import threading, time, torch, nest_asyncio, uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Any

app = FastAPI()

class ChatReq(BaseModel):
    model: str | None = None
    messages: list[dict[str, Any]]
    max_tokens: int | None = 512
    temperature: float | None = 0.2
    top_p: float | None = 0.9

@app.get('/healthz')
def healthz():
    return {'status': 'ok'}

@app.post('/v1/chat/completions')
def chat(r: ChatReq):
    ids = tokenizer.apply_chat_template(
        r.messages, tokenize=True, add_generation_prompt=True,
        return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids=ids,
            max_new_tokens=r.max_tokens or 512,
            temperature=max(r.temperature or 0.2, 1e-3),
            top_p=r.top_p or 0.9,
            do_sample=(r.temperature or 0.2) > 0,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
    return {'choices': [{'message': {'role': 'assistant', 'content': text}}]}

nest_asyncio.apply()
threading.Thread(
    target=lambda: uvicorn.run(app, host='0.0.0.0', port=8000, log_level='error'),
    daemon=True).start()
time.sleep(4)
print('shim up on :8000')

## 5. Open the public tunnel — KEEP THIS CELL RUNNING

Copy the `https://….trycloudflare.com` URL it prints. Locally set
`configs/model.yaml::llm.vllm_base_url` to `<that URL>/v1`, then run
`uv run python scripts/run_eval.py --task physics --with-llm` (and
`--task logic`). Stop this cell when done (it kills the tunnel).

In [ ]:
import subprocess, sys

proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
shown = False
for line in proc.stdout:
    if (not shown) and ('trycloudflare.com' in line):
        for tok in line.replace(chr(124), ' ').split():
            if tok.startswith('https://') and 'trycloudflare.com' in tok:
                print('=' * 60)
                print('PUBLIC URL  ->  ' + tok + '/v1')
                print('set configs/model.yaml::llm.vllm_base_url to that')
                print('keep this cell RUNNING; Stop it to close the tunnel')
                print('=' * 60)
                shown = True
                break
    sys.stdout.flush()